# Autoload & 1D Barcode Reader

| Summary | Image |
|--------|--------|
| <ul style="font-size:15px; line-height:1.6; margin-top:0;"><li><b>Feature:</b> Autoload with integrated 1D barcode reader</li><li><b>Operation:</b> Automatically loads carriers and plates from the front-loading tray onto the STAR deck.</li><li><b>Barcode Recognition:</b> Scans 1D barcodes on carriers, plates, tube racks, and tip racks to confirm ID and placement.</li><li><b>Purpose:</b> Reduces manual deck verification by ensuring that physically loaded carriers match the expected layout.</li><li><b>Workflow Benefit:</b> Verification + Speeds up run setup through hands-free loading and automatic identification.</li><li><b>Best For:</b> High-throughput or frequently changing deck configurations.</li></ul> | <div style="width:320px; text-align:center;">![real_autoload](img/autoload/hamilton_star_autoload.png)<br><i>Figure: Hamilton STAR Autoload system</i></div> |

## Background: Architecture of Hamilton STAR(let) Autoload

![hamilton_star_overview](img/autoload/hamilton_autoload_overview.png)

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and a deck class. The autoload subsystem is exposed as `star.driver.autoload` -- an {class}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload` instance (or `None` if the hardware is not installed).

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  hamilton_96_tiprack_50uL,
  Cor_96_wellplate_360ul_Fb,
)

star = STAR()
await star.setup()

# assign a tip rack carrier
tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_carrier[1] = tip_rack = hamilton_96_tiprack_50uL(name="tip_rack")
star.deck.assign_child_resource(tip_carrier, rails=10)

# assign a plate carrier
plt_carrier = PLT_CAR_L5AC_A00(name="plt_carrier")
plt_carrier[0] = plate = Cor_96_wellplate_360ul_Fb(name="plt")
star.deck.assign_child_resource(plt_carrier, rails=30)

The new API methods on {class}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload` take a `carrier_end_rail` integer instead of a carrier object. Compute it from the carrier's rail position and its width in rails:

```
carrier_end_rail = carrier_start_rail + carrier_width_in_rails - 1
```

For a `TIP_CAR_480_A00` (6 rails wide) assigned at rails=10: `carrier_end_rail = 10 + 6 - 1 = 15`.

In [ ]:
tip_carrier_end_rail = 15  # rails=10, 6 rails wide -> 10 + 6 - 1 = 15
plt_carrier_end_rail = 35  # rails=30, 6 rails wide -> 30 + 6 - 1 = 35

```{note}
Two starting points are possible:
1. Carriers have been moved directly on the deck.
2. Carriers have been left on the loading tray.

Here we assume we're starting with (1) Carriers have been moved directly on the deck.
```

## Querying autoload state

Check whether the autoload module is installed by inspecting `star.driver.autoload`:

In [ ]:
star.driver.autoload is not None

True

Query the current track of the autoload carrier handler with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.request_track`:

In [ ]:
await star.driver.autoload.request_track()

54

Query the autoload module type with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.request_type`:

In [ ]:
await star.driver.autoload.request_type()

'ML-STAR with 1D Barcode Scanner'

---
## Sensing carriers

The autoload sled has a front-facing proximity sensor.
This sensor can be used to scan the entire loading tray to identify whether there are any carriers currently on the loading tray with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.request_presence_of_carriers_on_loading_tray`:

In [ ]:
await star.driver.autoload.request_presence_of_carriers_on_loading_tray()

[]

```{note}
The autoload detects *only* the right-most track occupied by a carrier!
```

Similarly, if you only want to investigate a single position this can be done with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.request_presence_of_single_carrier_on_loading_tray`:

In [ ]:
await star.driver.autoload.request_presence_of_single_carrier_on_loading_tray(
  track=44
)

False

The counterpart to checking for carriers on the loading tray is checking for presence of carriers on the liquid handler's deck with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.request_presence_of_carriers_on_deck`:

In [ ]:
await star.driver.autoload.request_presence_of_carriers_on_deck()

[15, 35]

Note that we have assigned the carriers based on the left-most track they are occupying but `request_presence_of_carriers_on_deck()` detects the right-most track of carriers:
- tip_carrier: left-most track = 10, 6 tracks wide, right-most track = 15
- plt_carrier: left-most track = 30, 6 tracks wide, right-most track = 35

```{note}
`.request_presence_of_carriers_on_deck()` is technically not an 'autoload' command.
It uses the presence sensors at the back of the STAR(let) deck to identify whether a carrier is present.
i.e. there is no autoload movement involved, and it therefore works for STAR(let)s without an integrated barcode sensor too.
```

Together these methods enable capturing a full picture of the state of carriers on your liquid handler.

## Moving autoload

Use {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.move_to_safe_z_position`, {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.move_to_track`, and {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.park` to control the autoload position.

In [ ]:
# Always ensure the carrier handler is safely tucked away during movement.
await star.driver.autoload.move_to_safe_z_position()

In [ ]:
await star.driver.autoload.move_to_track(30)

In [ ]:
await star.driver.autoload.park()

---
## Basic Load & Unload

Use {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.unload_carrier` and {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.load_carrier` to move carriers between the deck and the loading tray. Both methods take `carrier_end_rail` instead of a carrier object.

In [ ]:
await star.driver.autoload.unload_carrier(
  carrier_end_rail=tip_carrier_end_rail,
  park_after=False,
)

In [ ]:
await star.driver.autoload.load_carrier(
  carrier_end_rail=tip_carrier_end_rail,
  carrier_barcode_reading=True,
  barcode_reading=True,
  park_autoload_after=False,
)

{'carrier_barcode': Barcode(data='08T0241707', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 'container_barcodes': None}

### Command Architecture for (Un)Loading & Barcode Reading

From a firmware-perspective a carrier can be in 3 "states":
1. On the deck
2. On the autoload belt
3. On the loading tray

{class}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload` enables easy transfer between these 3 states:

![hamilton_star_overview](img/autoload/hamilton_autoload_state_transfer.png)

```{note}
We could not find a command that enables the movement sequence: deck > autoload_belt > loading_tray yet.
`unload_carrier_after_barcode_scanning()` requires `load_carrier_from_tray_and_scan_carrier_barcode()` to precede it.
```

## 1D Barcode reading

There are 2 main types of 1D barcodes on a classic STAR(let) deck:
1. Carrier barcodes (orientation=vertical; located at the right-back of the carrier)
2. Container barcodes:
    - TipRack barcodes (orientation=horizontal)
    - Plate barcodes (orientation=horizontal)
    - Tube barcodes (orientation=vertical)

### 1D Barcode Symbologies

All barcodes must follow a standard encoding scheme, i.e. a "symbology".
Before reading barcodes it is important to know what barcode symbology you are expecting to read!

The following barcode symbologies can be detected by the system:

In [ ]:
list(star.driver.autoload.barcode_1d_symbology_dict.keys())

['ISBT Standard',
 'Code 128 (Subset B and C)',
 'Code 39',
 'Codebar',
 'Code 2of5 Interleaved',
 'UPC A/E',
 'YESN/EAN 8',
 'ANY 1D']

For the highest reading safety Hamilton recommends to use barcode type `Code128 (subset B and C)`.

This is the default symbology:

In [ ]:
star.driver.autoload._default_1d_symbology

'Code 128 (Subset B and C)'

You can change the expected barcode symbology with {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.set_1d_barcode_type`:

In [ ]:
await star.driver.autoload.set_1d_barcode_type("ISBT Standard")

star.driver.autoload._default_1d_symbology

'ISBT Standard'

In [ ]:
await star.driver.autoload.set_1d_barcode_type("Code 128 (Subset B and C)")

star.driver.autoload._default_1d_symbology

'Code 128 (Subset B and C)'

### Loading with Barcode Reading

1D barcode reading via the autoload can only occur during carrier loading actions.
So let's first unload the carrier:

In [ ]:
await star.driver.autoload.unload_carrier(
  carrier_end_rail=tip_carrier_end_rail,
  park_after=False,
)

...and this time activate reading both (1) the carrier barcode and (2) the container barcodes:

In [ ]:
barcode_readings = await star.driver.autoload.load_carrier(
  carrier_end_rail=tip_carrier_end_rail,
  carrier_barcode_reading=True,
  barcode_reading=True,
  # barcode_symbology="Code 39",
  # barcode_reading_direction="horizontal",
  # no_container_per_carrier=5,
  park_autoload_after=False,
)

This returns a dictionary:

In [ ]:
barcode_readings

{'carrier_barcode': Barcode(data='08T0241707', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 'container_barcodes': {0: Barcode(data='18235938752776512151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
  1: Barcode(data='18235938752776527151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
  2: Barcode(data='18235938752776549151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
  3: None,
  4: Barcode(data='18235938752776513151', symbology='Code 128 (Subset B and C)', position_on_resource='right')}}

```{warning}
The default behaviour of the 1D barcode scanner is set to read the standard 5x tip- and plate-carrier positions.
If you are reading barcodes on a different carrier (e.g. tube or trough carrier) you will have to modify the default parameters.
```

![hamilton_star_overview](img/autoload/hamilton_autoload_correct_1d_barcode_height.png)

```{warning}
The 1D barcode scanner uses a (class 2) laser targeted to a fixed height.
This is especially important when reading horizontal barcodes.
The z-height of the laser (during horizontal) barcode reading is `z=219` or 119 mm above the deck surface.
If your 1D barcode is not precisely positioned at this height, the 1D barcode reader cannot read your barcode.
To facilitate this height, use "DWP" carriers/MFX plate_holders for plates >40 mm in `size_z`, and "MP" carriers/MFX plate_holders for plates ~15 mm in `size_z`.
```

---
## Advanced 1D Barcode Reading

In [ ]:
await star.driver.autoload.unload_carrier(
  carrier_end_rail=tip_carrier_end_rail,
  park_after=False,
)

**Loading tray -> Autoload Belt (Carrier Barcode Reading) -> Loading Tray**

Use {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.load_carrier_from_tray_and_scan_carrier_barcode` to scan the carrier barcode, then {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.unload_carrier_after_barcode_scanning` to return the carrier to the tray:

In [ ]:
await star.driver.autoload.load_carrier_from_tray_and_scan_carrier_barcode(
  carrier_end_rail=tip_carrier_end_rail,
  # barcode_position=4.3,  # mm
  # barcode_reading_window_width=38.0,  # mm
  # reading_speed=128.1,  # mm/sec
)

Barcode(data='08T0241707', symbology='Code 128 (Subset B and C)', position_on_resource='right')

In [ ]:
await star.driver.autoload.unload_carrier_after_barcode_scanning()

**Loading tray -> Autoload Belt (Carrier Barcode Reading) -> Deck (Container Barcode Reading)**

This is equivalent to {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.load_carrier` but split into separate components:

In [ ]:
await star.driver.autoload.load_carrier_from_tray_and_scan_carrier_barcode(
  carrier_end_rail=tip_carrier_end_rail,
)

Barcode(data='08T0241707', symbology='Code 128 (Subset B and C)', position_on_resource='right')

In [ ]:
reading = await star.driver.autoload.load_carrier_from_belt(
  barcode_reading=True,
  park_autoload_after=False,
  # barcode_reading_direction="horizontal",
  # barcode_symbology="Code 128 (Subset B and C)",
  # reading_position_of_first_barcode=63.0,  # mm
  # no_container_per_carrier=5,
  # distance_between_containers=96.0,  # mm
  # width_of_reading_window=38.0,  # mm
  # reading_speed=128.1,  # mm/secs
)
reading

{0: Barcode(data='18235938752776512151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 1: Barcode(data='18235938752776527151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 2: Barcode(data='18235938752776549151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 3: None,
 4: Barcode(data='18235938752776513151', symbology='Code 128 (Subset B and C)', position_on_resource='right')}

**Deck -> Autoload Belt -> Deck (with container barcode reading only)**

Use {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.take_carrier_out_to_belt` to move a carrier from the deck to the identification position, then {meth}`~pylabrobot.hamilton.liquid_handlers.star.autoload.STARAutoload.load_carrier_from_belt` to load it back with barcode reading:

In [ ]:
await star.driver.autoload.take_carrier_out_to_belt(
  carrier_end_rail=tip_carrier_end_rail,
)

In [ ]:
reading = await star.driver.autoload.load_carrier_from_belt(
  barcode_reading=True,
  park_autoload_after=False,
)
reading

{0: Barcode(data='18235938752776512151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 1: Barcode(data='18235938752776527151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 2: Barcode(data='18235938752776549151', symbology='Code 128 (Subset B and C)', position_on_resource='right'),
 3: None,
 4: Barcode(data='18235938752776513151', symbology='Code 128 (Subset B and C)', position_on_resource='right')}

In [ ]:
await star.driver.autoload.move_to_safe_z_position()

In [ ]:
await star.driver.autoload.park()

---
## Teardown

In [ ]:
await star.stop()